In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### **Data Reading**

In [0]:
df = spark.read.format('parquet')\
    .load("abfss://bronze@stretaildevwestindia.dfs.core.windows.net/products")

In [0]:
df.display()

In [0]:
df = df.drop("_rescued_data")
df.display()

In [0]:
df.createOrReplaceTempView("products")

### **Functions**

In [0]:
%sql
CREATE OR REPLACE FUNCTION retail.bronze.discount_func(p_price DOUBLE)
RETURNS DOUBLE
LANGUAGE SQL
RETURN p_price * 0.9

In [0]:
%sql
select product_id, price, retail.bronze.discount_func(price) as discounted_price
from products

In [0]:
df = df.withColumn("discounted_price",expr("retail.bronze.discount_func(price)"))
df.display()

In [0]:
%sql
CREATE OR REPLACE FUNCTION retail.bronze.upper_func(p_brand STRING)
RETURNS STRING
LANGUAGE PYTHON
AS
$$
    return p_brand.upper()
$$

In [0]:
%sql
SELECT product_id, brand, retail.bronze.upper_func(brand) as brand_upper
FROM products

In [0]:
df.write.format("delta").mode("overwrite")\
    .save("abfss://silver@stretaildevwestindia.dfs.core.windows.net/products")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retail.silver.products_silver
USING DELTA
LOCATION "abfss://silver@stretaildevwestindia.dfs.core.windows.net/products"